<a href="https://colab.research.google.com/github/derekmok/machine-vision-coursework/blob/main/Machine_Vision_Final_Lab_Model_Training_MP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boto3 -q
!pip install opencv-python torch numpy torchvision mediapipe

## Download the data

The data for this assignment has been made available and is downloadable to disk by running the below cell.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os

# Connect to S3 without authentication (public bucket)
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

bucket_name = 'prism-mvta'
prefix = 'training-and-validation-data/'
download_dir = './video-data'

os.makedirs(download_dir, exist_ok=True)

# List all objects in the S3 path
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

video_names = []

for page in pages:
    if 'Contents' not in page:
        print("No files found at the specified path! Go and complain to the TAs!")
        break

    for obj in page['Contents']:
        key = obj['Key']
        filename = os.path.basename(key)

        if not filename:
            continue

        video_names.append(filename)

        local_path = os.path.join(download_dir, filename)
        print(f"Downloading: {filename}")
        s3.download_file(bucket_name, key, local_path)

print("\n" + "="*50)
print("Downloaded videos:")
print("="*50)
for name in video_names:
    print(name)

print(f"\nTotal: {len(video_names)} files")

These videos are now available in the folder "video-data". You can click on the folder icon on the left-hand-side of this screen to see the videos in a file explorer.

# Create your Datasets and Dataloaders

Some example code for approaching the first *two* TODOs is given below just to get you started. No starter code is given for the third TODO.

Note, the below code is very rough skeleton code. Make no assumptions as to the correct manner to architect your model based on the structure of this code.

Please feel free to (if not encouraged to) change every single line of the below code (change it to best suit your chosen model architecture, in the next section).

### TODO 1 (This is mostly already done for you - Please see the v1 provided below)

Each video in the folder is prefixed by a number. That number corresponds to the number of distinct pushups visible in the video. Write code to iterate over each video in the folder, and extract the corresponding target associated with the video.

### TODO 2 (This is also mostly already done for you - Please see the v1 provided below)


Divide the data into training and validation sets.

Optionally, you can also create out your own test set to assess your performance.

### TODO 3

Any preprocessing or augmentation of your data which you deem required, should (probably) go here. You are also free to include your data-augmentation code later, though doing it before creating your dataloaders is probably a good idea.

If you complete this TODO, to maintain experimental hygiene, feel free to modify the code which was provided for TODOs 1 and 2.

In [1]:
# Here is a basic implementation of the above two TODOs. You can assume the first TODO is completed correctly.

# Please modify this code to suit you best, as you decide on your preferred model architecture.

# For example, below here we are padding every video to 1,000 frames. That may or may not be a good idea.


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import os
import cv2
import numpy as np
from torchvision import tv_tensors
from torchvision.transforms import v2
import mediapipe as mp


class VideoDataset(Dataset):
    """Dataset for extracting pose landmarks from videos using MediaPipe.
    
    Extracts the following landmarks per frame:
    - Left/Right Shoulder (indices 11, 12)
    - Left/Right Elbow (indices 13, 14)
    - Left/Right Wrist (indices 15, 16)
    - Left/Right Hip (indices 23, 24)
    
    Returns a tensor of shape (T, 24) where T is the number of frames
    and 24 = 8 landmarks × 3 coordinates (x, y, z).
    """

    # MediaPipe Pose landmark indices for the body parts of interest
    LANDMARK_INDICES = [
        11, 12,  # Left/Right Shoulder
        13, 14,  # Left/Right Elbow
        15, 16,  # Left/Right Wrist
        23, 24,  # Left/Right Hip
    ]

    def __init__(self, video_dir, transform=None):
        """Initialize the dataset.
        
        Args:
            video_dir: Path to directory containing video files.
            transform: Optional transform to apply to the landmark sequence.
        """
        self.video_dir = video_dir
        self.transform = transform

        self.video_files = [
            f for f in os.listdir(video_dir)
            if f.endswith(('.mp4', '.avi', '.mov'))
        ]

        self.labels = [
            int(f.split('_')[0]) for f in self.video_files
        ]

        # Initialize MediaPipe Pose
        self.mp_pose = mp.solutions.pose

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path = os.path.join(self.video_dir, self.video_files[idx])
        landmarks_sequence = self._extract_landmarks(video_path)
        label = self.labels[idx]

        if self.transform:
            landmarks_sequence = self.transform(landmarks_sequence)

        return landmarks_sequence, label

    def _extract_landmarks(self, path):
        """Extract pose landmarks from a video using MediaPipe.
        
        Args:
            path: Path to the video file.
            
        Returns:
            Tensor of shape (T, 24) containing landmark coordinates,
            where T is the number of frames and 24 = 8 landmarks × 3 (x, y, z).
            If no pose is detected in a frame, landmarks are set to 0.
        """
        cap = cv2.VideoCapture(path)
        landmarks_list = []

        with self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as pose:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                # Convert BGR to RGB for MediaPipe
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                # Process the frame
                results = pose.process(frame_rgb)

                # Extract landmarks for the frame
                frame_landmarks = []
                if results.pose_landmarks:
                    for idx in self.LANDMARK_INDICES:
                        landmark = results.pose_landmarks.landmark[idx]
                        frame_landmarks.extend([landmark.x, landmark.y, landmark.z])
                else:
                    # No pose detected, fill with zeros
                    frame_landmarks = [0.0] * (len(self.LANDMARK_INDICES) * 3)

                landmarks_list.append(frame_landmarks)

        cap.release()

        # Convert to tensor: shape (T, 24)
        landmarks_tensor = torch.tensor(landmarks_list, dtype=torch.float32)

        return landmarks_tensor


def collate_fn(batch):
    """Pad all landmark sequences to the maximum length in the batch.
    
    Args:
        batch: List of (landmarks, label) tuples where landmarks has shape (T, 24).
        
    Returns:
        landmarks_batch: Tensor of shape (B, max_seq_len, 24).
        labels_batch: Tensor of shape (B,).
        lengths: Tensor of shape (B,) containing original sequence lengths.
    """
    landmarks_list, labels = zip(*batch)

    # Find the maximum sequence length in this batch
    lengths = [landmarks.shape[0] for landmarks in landmarks_list]
    max_len = max(lengths)

    padded_landmarks = []
    for landmarks in landmarks_list:
        num_frames = landmarks.shape[0]
        if num_frames < max_len:
            # Pad with zeros: shape (max_len - num_frames, 24)
            padding = torch.zeros(max_len - num_frames, landmarks.shape[1])
            landmarks = torch.cat([landmarks, padding], dim=0)
        padded_landmarks.append(landmarks)

    landmarks_batch = torch.stack(padded_landmarks, dim=0)
    labels_batch = torch.tensor(labels)
    lengths_batch = torch.tensor(lengths)

    return landmarks_batch, labels_batch, lengths_batch


def get_dataloaders(video_dir, batch_size=4, val_split=0.2):
    """Create train and validation dataloaders for landmark sequences.
    
    Args:
        video_dir: Path to directory containing video files.
        batch_size: Number of samples per batch.
        val_split: Fraction of data to use for validation.
        
    Returns:
        train_loader: DataLoader for training data.
        val_loader: DataLoader for validation data.
    """

    full_dataset = VideoDataset(video_dir)

    val_size = int(len(full_dataset) * val_split)
    train_size = len(full_dataset) - val_size

    train_dataset, val_dataset = random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn
    )

    print(f"Train: {len(train_dataset)} videos, Val: {len(val_dataset)} videos\n")

    return train_loader, val_loader


video_dir = './video-data'

train_loader, val_loader = get_dataloaders(video_dir, batch_size=4, val_split=0.2)

for landmarks, labels, lengths in train_loader:
    print(f"Landmarks shape: {landmarks.shape}")  # (B, max_seq_len, 24)
    print(f"Labels: {labels}")
    print(f"Sequence lengths: {lengths}")
    break

AttributeError: module 'mediapipe' has no attribute 'solutions'

# Create a Model

For this assignment, we request you use PyTorch. Below is an example of how to instantiate a very basic PyTorch model.

Note, this model below needs a _lot_ of work.

Please include your code for creating your model below.

The only constraint here is that you define a Python object which inherits from a PyTorch nn.Module object. Beyond that, please feel free to implement anything you like: Transformer, Vision Transformer, MLP, CNN, etc.

### TODO 4

Create your model.

In [ ]:

import torch
import torch.nn as nn
from huggingface_hub import HfApi, hf_hub_download


class SimpleVideoClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Average over frames, then use a simple CNN
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (B, C, T, H, W)
        # Average over time dimension
        x = x.mean(dim=2)  # (B, C, H, W)
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x



# Train your Model

### TODO 5

Training time! Please include your training code below.

As per above, please feel free (and encouraged) to rip out all of the below code and replace with your (much better) code.

The below should just be used as an example to get you started.

In [ ]:
import torch.optim as optim


def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

    return total_loss / len(train_loader), correct / total


def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(test_loader), correct / total


def train_model(epochs=5, lr=1e-3):
    """Train and return your model."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    model = SimpleVideoClassifier().to(device)
    print("Instantiated model.\n")
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_loader, val_loader = get_dataloaders(video_dir='./video-data')
    print("Got dataloaders.\n")

    print("Go time. Let the training commence.\n")

    for epoch in range(1, epochs + 1):

        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return model



In [ ]:
# setting a manual seed for reproducibility
torch.manual_seed(0)
model = train_model()

# Evaluation

## TODO 6

Include any code which you feel is useful for evaluating your model performance below.

In [ ]:
# YOUR CODE HERE

# Hugging Face

It is a requirement of this assignment that you submit your trained model to a repo on Hugging Face, and make it publicly available. Below, we provide code which should help you do this.

## TODO 7

Upload your model to HuggingFace

Install the dependencies:

In [ ]:
!pip install huggingface_hub

You'll now need to log in to Hugging Face via the command line. To do this, you'll need to generate a token on your Hugging Face account. To generate a token, run the below command, and click on the link which appears.

In [ ]:
!hf auth login

The below code will only run if you have already trained a model with variable name 'model'.

The below code will take your trained model, and upload it to a *public* HuggingFace repo in your account called "mv-final-assignment".

(Note - in this example, we have set 'private=False' in the upload_to_hub method. This makes your model public).

You should double-check that your model is in fact public. To do that, you can navigate (in an incognito tab, in a browser) to https://huggingface.co/YOUR_USERNAME/YOUR_MODEL_NAME and see if that page loads. If your model is public, it will. (Simply being able to run the below code will not guarantee that your model is in fact public, because, you have now authenticated yourself with the huggingface CLI).

In [ ]:
# YOUR HUGGING FACE USERNAME BELOW
hf_username = 'rossamurphy'

In [ ]:
import torch
import torch.nn as nn
from huggingface_hub import HfApi, hf_hub_download


def save_model(model, path="model.pt"):
    """Save the model weights to a file."""
    torch.save(model.state_dict(), path)
    print(f"Model saved to {path}")


def upload_to_hub(local_path="model.pt", repo_id=f"{hf_username}/mv-final-assignment"):
    """
    Upload model to Hugging Face Hub.

    Args:
        local_path: Path to your saved model file
        repo_id: Your repo in format "username/model-name"
    """
    api = HfApi()

    # Create the repo first (if it already exists, this will just skip)
    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        exist_ok=True,  # Don't error if it already exists
        private=False,  # Make it public so TAs can access
    )

    # Now upload the file
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo="model.pt",
        repo_id=repo_id,
        repo_type="model",
    )

    print(f"Model uploaded to https://huggingface.co/{repo_id}")


# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":

    save_model(model, "mv-final-assignment.pt")

    upload_to_hub("mv-final-assignment.pt", f"{hf_username}/mv-final-assignment")